# 🪰 RAPID v3 — Fly-CL on CIFAR-100
> **Block-wise Absolute WTA · NCM Prototype Classifier · Fisher-adaptive Projection**

| Param | Value |
|---|---|
| Dataset | CIFAR-100 (100 classes / 10 tasks) |
| Backbone | ViT-B/16 (frozen) |
| Projection | Growable Sparse Random (expand_dim=10000) |
| Classifier | NCM + Per-class Subspace Cosine |
| Normalization | [-1, 1] (Paper Appendix C.3) |

---
Run all cells top-to-bottom. Results appear in **Cell 6** and **Cell 7**.

## ⚙️ Step 1 — Install dependencies

In [ ]:
# timm: ViT backbone loader
!pip install timm==0.9.16 -q
print("✅ Dependencies ready.")

## 📦 Step 2 — Clone / update repo

In [ ]:
import os

REPO_URL = "https://github.com/ZaPhat206/LAB_FLY.git"
WORK_DIR = "/kaggle/working/LAB_FLY"

print("Cleaning up old repo...")
!rm -rf {WORK_DIR}

print("Cloning fresh repo...")
!git clone {REPO_URL} {WORK_DIR} -q

os.chdir(WORK_DIR)
print(f"✅ Working dir: {os.getcwd()}")
print("   Latest Commit:")
!git log -1 --format="      %h | %cd | %s"


## 🧠 Step 3 — Download pretrained ViT-B/16

In [ ]:
import os
os.chdir(f"{WORK_DIR}/pretrained_model")
!sh download.sh
os.chdir(WORK_DIR)
print("✅ Pretrained model ready.")

## 🔧 Step 4 — Fix Kaggle package conflict

In [ ]:
# Kaggle pre-installs HuggingFace 'datasets' → rename our folder to avoid import conflict
import os, shutil
os.chdir(WORK_DIR)

if os.path.isdir("datasets") and not os.path.isdir("my_datasets"):
    shutil.move("datasets", "my_datasets")
    print("Renamed: datasets → my_datasets")
else:
    print("Rename: already done or not needed.")

!sed -i 's/from datasets.load_dataset/from my_datasets.load_dataset/g' main.py
!find my_datasets -name '*.py' -exec sed -i 's/from datasets/from my_datasets/g' {} + 2>/dev/null || true
print("✅ Import conflict fixed.")

## 📂 Step 5 — Link CIFAR-100 dataset
> **Prerequisite:** Add your uploaded CIFAR-100 dataset via the **Data tab** (right panel).  
> The cell below auto-detects its path — no internet download needed.

In [ ]:
import os, glob, builtins

# Auto-detect CIFAR-100 inside any added Kaggle dataset
# torchvision expects: root/cifar-100-python/{meta, train, test}
matches = glob.glob("/kaggle/input/*/cifar-100-python")

if matches:
    CIFAR_PYTHON_DIR = matches[0]
    CIFAR_ROOT       = os.path.dirname(CIFAR_PYTHON_DIR)
    builtins.CIFAR_ROOT = CIFAR_ROOT
    print(f"✅ CIFAR-100 found  : {CIFAR_PYTHON_DIR}")
    print(f"   Will pass --root : {CIFAR_ROOT}")
    # Quick sanity check
    for f in ["meta", "train", "test"]:
        path = os.path.join(CIFAR_PYTHON_DIR, f)
        status = "✅" if os.path.exists(path) else "❌ MISSING"
        print(f"   {status}  {path}")
else:
    print("⚠️  CIFAR-100 dataset NOT found in /kaggle/input/")
    print("   → Go to the Data tab → Add Data → search your dataset.")
    print("   → Falling back to auto-download (slow, ~161 MB).")
    builtins.CIFAR_ROOT = "../data"

## 🚀 Step 6 — Run RAPID v3

In [ ]:
# --- [ CELL 6 ] --- Lệnh chạy Training ---
import subprocess
import sys

cifar_root = "/kaggle/input/cifar-100/cifar-100-python"

CMD = [
    sys.executable, "main.py",
    "--dataset",           "CIFAR-100",
    "--root",              cifar_root,
    "--num_classes",       "100",
    "--num_tasks",         "10",
    "--model_name",        "vit_base_patch16_224",
    "--embedding_dim",     "768",
    "--expand_dim",        "10000",
    "--synaptic_degree",   "300",
    "--coding_level",      "0.01",
    
    # 🔥 Combo 1: Ridge + Subspace + Procrustes
    "--classifier_type",   "ridge_subspace",
    "--use_subspace", 
    "--use_procrustes",
    
    "--fisher_block",      "512",
    "--fisher_sat",        "0.005",
    "--data_augmentation", "vit",
    "--seed",              "1993",
    "--batch_size",        "128",
    "--gpu",               "0",
]

print("Running command:")
print(" ".join(CMD))

# Chạy sub-process và in log trực tiếp
process = subprocess.Popen(CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in iter(process.stdout.readline, ''):
    print(line, end='')

process.wait()


## 📊 Step 7 — Results summary

In [ ]:
# ── Cell 7: Parse & display results ───────────────────────
import re, numpy as np

with open("/kaggle/working/rapid_cifar_output.txt") as f:
    txt = f.read()

# ── Parse Accuracy Matrix ──────────────────────────────────
acc_matrix = []
in_block = False
for line in txt.splitlines():
    if "Accuracy Matrix" in line:
        in_block = True; continue
    if in_block:
        l = line.strip()
        if l.startswith("["):
            try:    acc_matrix.append(eval(l))
            except: pass
        elif l == "" and acc_matrix:
            in_block = False

# ── Parse scalar metrics ───────────────────────────────────
def grab(pattern):
    m = re.search(pattern, txt, re.DOTALL)
    return float(m.group(1)) if m else None

accumulated_acc = grab(r"Accumulated Accuracy\s+([\d.]+)")
avg_train_time  = grab(r"Average Training Time\s+([\d.]+)")
avg_feat_time   = grab(r"Average Feature Extract Time\s+([\d.]+)")

# ── Forgetting (BWT) ──────────────────────────────────────
forgetting_list = []
if acc_matrix:
    T = len(acc_matrix)
    for i in range(T - 1):
        row   = acc_matrix[i]
        peak  = float(row[i])   if isinstance(row[i],   (int, float)) else 0.0
        final = float(row[T-1]) if isinstance(row[T-1], (int, float)) and str(row[T-1]) != "0.00" else 0.0
        if peak > 0:
            forgetting_list.append(peak - final)
mean_forgetting = float(np.mean(forgetting_list)) if forgetting_list else 0.0

# ── Print summary ─────────────────────────────────────────
print("╔" + "═"*50 + "╗")
print(f"║  {'🪰 RAPID v3  —  CIFAR-100  (10 tasks)':^46}  ║")
print("╠" + "═"*50 + "╣")
print(f"║  {'Accumulated Accuracy':<28} {accumulated_acc if accumulated_acc else 0:>8.2f} %    ║")
print(f"║  {'Mean Forgetting (BWT)':<28} {mean_forgetting:>8.2f} %    ║")
print(f"║  {'Avg Training Time / task':<28} {avg_train_time if avg_train_time else 0:>8.2f} s    ║")
print(f"║  {'Avg Feature Extract Time':<28} {avg_feat_time if avg_feat_time else 0:>8.2f} s    ║")
print("╚" + "═"*50 + "╝")

# ── Accuracy Matrix ───────────────────────────────────────
if acc_matrix:
    T = len(acc_matrix)
    print()
    header = "Task  │ " + " │ ".join(f"T{j+1:02d}" for j in range(T))
    print(header)
    print("─" * len(header))
    for i, row in enumerate(acc_matrix):
        cells_str = []
        for j, v in enumerate(row):
            if isinstance(v, (int, float)) and str(v) != "0.00":
                cells_str.append(f"{float(v):5.2f}")
            else:
                cells_str.append("  —  ")
        print(f"  T{i+1:02d} │ " + " │ ".join(cells_str))


## 📈 Step 8 — Visualize (optional)

In [ ]:
# ── Cell 8: Plot ──────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import re

# Parse A_t from "Average Accuracy\n XX, XX, ..."
avg_accs = []
m = re.search(r"Average Accuracy\s+([\d.,\s]+)", txt)
if m:
    vals = m.group(1).split(",")
    for v in vals:
        v = v.strip()
        if v:
            try: avg_accs.append(float(v))
            except: pass

if not avg_accs:
    print("No A_t data found. Check output file.")
else:
    tasks = list(range(1, len(avg_accs) + 1))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("RAPID v3 — CIFAR-100 (10 tasks)", fontsize=15, fontweight="bold", y=1.02)
    fig.patch.set_facecolor("#0f0f1a")
    for ax in axes:
        ax.set_facecolor("#1a1a2e")
        ax.tick_params(colors="white")
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        ax.title.set_color("white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444")

    # ── Plot 1: Average Accuracy ──
    axes[0].plot(tasks, avg_accs, color="#00d4ff", linewidth=2.5,
                 marker="o", markersize=8, markerfacecolor="#ff6b6b", label="RAPID v3")
    if accumulated_acc:
        axes[0].axhline(y=accumulated_acc, linestyle="--", color="#ffd700", linewidth=1.5,
                        label=f"A_bar = {accumulated_acc:.2f}%")
    axes[0].set_xlabel("Task", fontsize=12)
    axes[0].set_ylabel("Average Accuracy (%)", fontsize=12)
    axes[0].set_title("Incremental Avg Accuracy (Aₜ)", fontsize=13)
    axes[0].legend(facecolor="#2a2a3e", labelcolor="white")
    axes[0].grid(True, alpha=0.2, color="white")
    axes[0].set_xticks(tasks)
    axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f"))

    # ── Plot 2: Forgetting per task ──
    if forgetting_list:
        tids  = list(range(1, len(forgetting_list) + 1))
        colors = ["#ff6b6b" if v > mean_forgetting else "#4ecdc4" for v in forgetting_list]
        axes[1].bar(tids, forgetting_list, color=colors, alpha=0.85, edgecolor="#222")
        axes[1].axhline(y=mean_forgetting, linestyle="--", color="#ffd700", linewidth=1.5,
                        label=f"Mean = {mean_forgetting:.2f}%")
        axes[1].set_xlabel("Task", fontsize=12)
        axes[1].set_ylabel("Forgetting (%)", fontsize=12)
        axes[1].set_title("Per-task Forgetting (BWT)", fontsize=13)
        axes[1].legend(facecolor="#2a2a3e", labelcolor="white")
        axes[1].grid(True, alpha=0.2, axis="y", color="white")
        axes[1].set_xticks(tids)

    plt.tight_layout()
    out_img = "/kaggle/working/rapid_cifar_results.png"
    plt.savefig(out_img, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"✅ Plot saved → {out_img}")
